In [1]:
import torch 
import numpy as np 
import h5py
import os
from pathlib import Path
import importlib
import IPython.display as ipd

import src.spatial_attn_lightning as binaural_lightning 
import yaml
from pytorch_lightning import Trainer, seed_everything

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

torch.set_float32_matmul_precision('medium')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [3]:

config_path = "config/binaural_attn/word_task_standard_v06.yaml"
# config_path = "config/binaural_attn/word_task_half_co_loc_v06.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 2
config['hparas']['batch_size'] = 32
config['audio']['rep_kwargs']['rep_on_gpu'] = True
print(f"Default lr is {config['hparas']['lr']}")
config['hparas']['lr'] = 0.000001
print(f"Trying lr = {config['hparas']['lr']}")



Default lr is 0.0001
Trying lr = 1e-06


In [4]:
## func to init layers with he initialization 
def kaiming_init(model, mode='fan_in', init_type='normal'):
    if init_type=='normal':
        init_fn = torch.nn.init.kaiming_normal_
    elif init_type=='uniform':
        init_fn = torch.nn.init.kaiming_uniform_

    for name, param in model.named_parameters():
        if 'attn' in name:
            continue
        if name.endswith(".bias"):
            # print(f"zero init {name}")
            param.data.fill_(0)
        elif any( part in name for part in ['conv', 'fullyconnected']):
            # print(f"kaiming init {name}")
            init_fn(param, mode=mode, nonlinearity='relu')



In [5]:
seed_everything(0)
importlib.reload(binaural_lightning)
module = binaural_lightning.BinauralAttentionModule(config)

[rank: 0] Seed set to 0


num_classes={'num_words': 800}
Model performing word task
center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torchaudio/functional/functional.py:1371: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  warnings.warn(


In [6]:
# kaiming_init(module, mode='fan_in', init_type='uniform')

In [7]:
trainer = Trainer(
    precision="32",
    limit_val_batches=0.0,
    num_nodes=1,
    # benchmark=True,
    devices=1, # was gpus=1,
    # detect_anomaly=True,
    # strategy="ddp_notebook",
    accelerator="gpu",
)
trainer.fit(module)

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/lightning_fabric/plugins/environments/slurm.py:191: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /om2/user/imgriff/conda_envs/pytorch_2/lib/python3.1 ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/pytorch_lightning/loops/utilities.py:73: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type                         | Params
------------------------------------------------------------------
0 | audio_transforms | AudioCompose                 | 0     
1 | model            | OptimizedModule      

Using v06 dataset
/om/scratch/Sun/imgriff/datasets/spatial_audio_pipeline/assets/dataset_binaural_attn/v06
cue type: voice_and_location
985 files in train concat dataset
len training set = 123125
Epoch 0:   0%|          | 97/123125 [05:24<114:10:15,  0.30it/s, v_num=3.47e+7, train_loss=6.660, grad_norm=5.410]

Process ForkProcess-1:
Process ForkProcess-2:
Traceback (most recent call last):
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/concurrent/futures/process.py", line 244, in _process_worker
    call_item = call_queue.get(block=True)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/multiprocessing/queues.py", line 103, in get
    res = self._recv_bytes()
          ^^^^^^^^^^^^^^^^^^
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/multiprocessing/connection.py", line 215, in recv_bytes
    buf = self._recv_bytes(maxlength)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/multi

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/pytorch_lightning/trainer/call.py:54: Detected KeyboardInterrupt, attempting graceful shutdown...


In [ ]:
## Check dataset 

dataset = module.dataset(**config['corpus'], mode='train')

In [ ]:
cue, target, distractor, labels = dataset[0]

In [ ]:
aud_transforms = module.audio_transforms

In [ ]:
ix = 1080

cue, target, distractor, labels = dataset[ix]

scene, _ = aud_transforms(target, distractor)

print("cue")
ipd.display(ipd.Audio(cue[0], rate=44100, normalize=False))
print("target")
ipd.display(ipd.Audio(target[0], rate=44100, normalize=False))
print("distractor")
ipd.display(ipd.Audio(distractor[0], rate=44100, normalize=False))
print("scene")
ipd.display(ipd.Audio(scene[0], rate=44100, normalize=False))